# Route Map Generation

This notebook generates an interactive map that highlights accidents and fines located near a selected route.

## Inputs

- `data/processed/trips/gemma/eclipse.csv`
- `data/processed/acidentes_processado.csv`
- `data/processed/multas_processado.csv`

## Outputs

- `outputs/reports/route_accidents_fines_map.html`

## Repository Paths

All paths are resolved from the repository root with `pathlib.Path`, so the notebook can run from either the project root or the `notebooks/` directory.


In [ ]:
from pathlib import Path

import folium
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "data").exists():
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

DATA_PROCESSED_DIR = REPO_ROOT / "data" / "processed"
TRIPS_DIR = DATA_PROCESSED_DIR / "trips"
OUTPUTS_REPORTS_DIR = REPO_ROOT / "outputs" / "reports"
OUTPUTS_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

route = pd.read_csv(TRIPS_DIR / "gemma" / "eclipse.csv")
acc = pd.read_csv(DATA_PROCESSED_DIR / "acidentes_processado.csv")
mul = pd.read_csv(DATA_PROCESSED_DIR / "multas_processado.csv")

route_points = route[["latitude", "longitude"]].dropna()
route_points = route_points[(route_points["latitude"] != 0) & (route_points["longitude"] != 0)].copy()

center_lat = float(route_points["latitude"].mean())
center_lon = float(route_points["longitude"].mean())


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))


def min_dist_to_route_km(lat_series, lon_series, route_array, chunk=2000):
    lats = lat_series.to_numpy()
    lons = lon_series.to_numpy()
    mins = np.empty(len(lats))
    for index in range(0, len(lats), chunk):
        slice_obj = slice(index, index + chunk)
        lat = lats[slice_obj][:, None]
        lon = lons[slice_obj][:, None]
        dist = haversine_km(lat, lon, route_array[None, :, 0], route_array[None, :, 1])
        mins[slice_obj] = dist.min(axis=1)
    return mins


route_array = route_points[["latitude", "longitude"]].to_numpy()
radius_center_km = 3.0
max_dist_route_km = 0.30

acc_near = acc[haversine_km(acc["latitude"], acc["longitude"], center_lat, center_lon) <= radius_center_km].copy()
mul_near = mul[haversine_km(mul["latitude"], mul["longitude"], center_lat, center_lon) <= radius_center_km].copy()

acc_near["dist_route_km"] = min_dist_to_route_km(acc_near["latitude"], acc_near["longitude"], route_array, chunk=500)
mul_near["dist_route_km"] = min_dist_to_route_km(mul_near["latitude"], mul_near["longitude"], route_array, chunk=2000)

acc_near = acc_near[acc_near["dist_route_km"] <= max_dist_route_km].copy()
mul_near = mul_near[mul_near["dist_route_km"] <= max_dist_route_km].copy()

route_map = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="OpenStreetMap")

folium.PolyLine(
    route_points[["latitude", "longitude"]].values.tolist(),
    color="#c2185b",
    weight=9,
    opacity=0.95,
).add_to(route_map)

accidents_group = folium.FeatureGroup(name="Accidents", show=True)
for _, row in acc_near.iterrows():
    popup = folium.Popup(
        f"<b>Accident</b><br>"
        f"<b>Date:</b> {row.get('data', '')}<br>"
        f"<b>Time:</b> {row.get('hora', '')}<br>"
        f"<b>Type:</b> {row.get('tipo', '')}<br>"
        f"<b>Severity:</b> {row.get('gravidade', '')}<br>"
        f"<b>Distance to route:</b> {row['dist_route_km'] * 1000:.0f} m",
        max_width=300,
    )
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        icon=folium.Icon(icon="exclamation-triangle", prefix="fa", color="orange"),
        popup=popup,
    ).add_to(accidents_group)
accidents_group.add_to(route_map)

fines_group = folium.FeatureGroup(name="Fines", show=False)
for _, row in mul_near.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        icon=folium.Icon(icon="gavel", prefix="fa", color="blue"),
    ).add_to(fines_group)
fines_group.add_to(route_map)

folium.LayerControl(collapsed=False).add_to(route_map)

output_path = OUTPUTS_REPORTS_DIR / "route_accidents_fines_map.html"
route_map.save(output_path)
print(f"Saved report to {output_path.relative_to(REPO_ROOT)}")
